# Translation V2: اختبار التطوير الصغير (Development Smoke Test)

**الهدف:** نتأكد إن V2 بيشتغل هندسيًا من البداية للنهاية على بيئة حقيقية. **هاد مو تقييم لجودة الترجمة.**

- 8 وحدات متتالية من المحاضرة الحالية (`tu_0017` لـ `tu_0024`)، وكل وحدة معها سياقها الأصلي.
- مساحة عمل وكاش وذاكرة مصطلحات معزولة باسم `dev_smoke_test`، وذاكرة الكورس الفعلية ما بتنلمس.
- سقف واضح لطلبات Gemini: 60 طلب للتشغيل الأول، و0 لإعادة الإنتاج.
- الكود مثبّت على commit واحد (`COMMIT` بالخلية 1)، مو على فرع بيتغيّر.

**التشغيل:**
1. Runtime → Change runtime type → **T4 GPU**.
2. أضف `GEMINI_API_KEY` في Secrets (أيقونة المفتاح) وفعّل Notebook access.
3. الصق الـ SHA الكامل بـ `COMMIT` في الخلية 1.
4. شغّل الخلايا بالترتيب. إذا فشلت خلية، وقّف وابعت رسالة الخطأ.

In [ ]:
# 1) Drive + the code pinned to one commit + requirements  (~3 min)
COMMIT = ''   # paste the full 40-character SHA you were given
assert len(COMMIT) == 40, 'paste the full commit SHA in COMMIT'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

LESSON = 'lesson_bfe775aa22957d44_academic_v2'
SUB = Path('outputs') / 'academic_fresh_pipeline' / LESSON
_roots = [Path('/content/drive/MyDrive/ArabicDubbingV2')]
_roots += sorted(Path('/content/drive/.shortcut-targets-by-id').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/ArabicDubbingV2'))
_roots += sorted(Path('/content/drive').glob('*/*/ArabicDubbingV2'))
RUN_DIR = next((r / SUB for r in _roots if (r / SUB).exists()), None)
if RUN_DIR is None:
    raise FileNotFoundError('ArabicDubbingV2 is not visible: add a shortcut to it in My Drive (same account), then rerun.')
WORK_SUBDIR = 'translation_step_dev_smoke_test'          # isolated: own runs, caches and memory
WORK = RUN_DIR / WORK_SUBDIR
WORK.mkdir(parents=True, exist_ok=True)
(WORK / '.write_test').write_text('ok'); (WORK / '.write_test').unlink()
print('RUN_DIR =', RUN_DIR)
print('WORK    =', WORK)

TR_REPO = Path('/content/translation_step_repo')
if not TR_REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git', str(TR_REPO)], check=True)
subprocess.run(['git', '-C', str(TR_REPO), 'fetch', '-q', 'origin', 'translation-v2-language'], check=True)
subprocess.run(['git', '-C', str(TR_REPO), 'checkout', '-q', '--detach', COMMIT], check=True)
HEAD = subprocess.run(['git', '-C', str(TR_REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert HEAD == COMMIT, f'checked out {HEAD}, expected {COMMIT}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(TR_REPO / 'translation' / 'requirements-colab.txt')], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'hunspell', 'hunspell-ar'], check=True, capture_output=True)
sys.path.insert(0, str(TR_REPO / 'translation'))
print('code at', HEAD)

In [ ]:
# 2) secrets (by hand: Colab asks for access the first time). The key is never printed.
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') or ''
print('GEMINI_API_KEY loaded:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
# 3) environment: PyTorch / CUDA / bitsandbytes on this GPU, then the preflight (no model is loaded here)
import importlib.metadata as md
import torch
print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> T4 GPU'
p = torch.cuda.get_device_properties(0)
print('GPU', p.name, f'| {p.total_memory / 1e9:.1f} GB')
import bitsandbytes as bnb
print('bitsandbytes', md.version('bitsandbytes'), '| transformers', md.version('transformers'))
_layer = bnb.nn.Linear4bit(64, 64, compute_dtype=torch.float16, quant_type='nf4').cuda()   # tiny 4-bit check
print('4-bit layer on GPU ok:', tuple(_layer(torch.randn(2, 64, dtype=torch.float16, device='cuda')).shape))
del _layer

from dubbing_translation.io import V2_CONFIG, load_config
from dubbing_translation.preflight import preflight, requirements_for
HF_MODELS = [('tencent/Hy-MT2-7B', 'hy_mt2_terms', 1)]
PREFLIGHT = preflight(requirements_for(use_hymt=True, hf_models=HF_MODELS, use_llm=True, refine=True, rank=True,
                                       language=True), write_dir=WORK)
print('preflight ok | versions:', PREFLIGHT['versions'])

In [ ]:
# 4) the smoke-test units: 8 consecutive units of the lecture, each with its original context
#    (a safe merge, two suspected fragments, spelled-out numbers, notation, regression case tu_0024)
import json
from dubbing_translation.colab import select_units
from dubbing_translation.io import load_tasks
UNITS = [f'tu_{i:04d}' for i in range(17, 25)]
SMOKE = select_units(load_tasks(RUN_DIR / 'experiments_v1' / 'tasks.json'), UNITS)
for t in SMOKE:
    print(t['unit_id'], f"{t.get('source_duration_s', 0):5.1f}s |", t['source_text'][:90])
COURSE_MEMORY = WORK / 'course_memory' / 'dev_smoke_test.json'    # NOT the real course memory
MAX_LLM_CALLS = 60                                                 # Gemini cap for run 1 (free tier only)

In [ ]:
# 5) run 1: the whole V2 pipeline on the 8 units (~15-25 min on a T4)
import time
from dubbing_translation import colab as trc
from dubbing_translation.io import V2_CONFIG

def wait(job, name):
    while not job['done']:
        print(time.strftime('%H:%M:%S'), '|', job['log'][-1] if job['log'] else 'starting...', flush=True)
        time.sleep(30)
    (WORK / 'smoke_logs').mkdir(exist_ok=True)
    (WORK / 'smoke_logs' / f'{name}.log').write_text('\n'.join(job['log']) + '\n\n' + str(job['error'] or ''),
                                                    encoding='utf-8')
    trc.status(job, tail=8)
    if job['error']:
        raise RuntimeError('the run failed: send smoke_logs/' + name + '.log')
    r = job['result']
    print('LLM requests:', r['llm_requests'], '/', r['llm_request_cap'], '| GPU peak GB:', r['gpu_peak_gb'])
    return r

JOB1 = trc.start(RUN_DIR, work_subdir=WORK_SUBDIR, config=V2_CONFIG, use_hymt=True, hf_models=HF_MODELS,
                 use_llm=True, refine=True, rank=True, label='smoke', course_memory=COURSE_MEMORY,
                 units=UNITS, max_llm_calls=MAX_LLM_CALLS)
R1 = wait(JOB1, 'run1_smoke')
RUN1 = Path(R1['run_dir'])
print('run 1:', RUN1)

In [ ]:
# 6) engineering checks of run 1 (items 5-11). Not a quality evaluation.
CHECK = trc.smoke_check(RUN1, expected_units=UNITS, expected_commit=COMMIT, course_memory=COURSE_MEMORY)
for name, c in CHECK.items():
    print('OK  ' if c['ok'] else 'FAIL', name, '|', json.dumps(c['detail'], ensure_ascii=False, default=str)[:300])

In [ ]:
# 7) run 2: replay with the SAME memory snapshot and the same caches, with 0 LLM requests allowed.
#    Any new model call would stop the run (degraded) instead of being made.
JOB2 = trc.start(RUN_DIR, work_subdir=WORK_SUBDIR, config=V2_CONFIG, use_hymt=True, hf_models=HF_MODELS,
                 use_llm=True, refine=True, rank=True, label='smoke_replay', units=UNITS, max_llm_calls=0,
                 course_memory_snapshot=RUN1 / 'course_memory_snapshot.json')
R2 = wait(JOB2, 'run2_replay')
REPLAY = trc.replay_check(RUN1, Path(R2['run_dir']))
print(json.dumps(REPLAY, ensure_ascii=False, indent=1))

In [ ]:
# 8) one zip with what to send back (stays on your Drive; nothing is uploaded anywhere)
import shutil, zipfile
stamp = time.strftime('%Y%m%d_%H%M%S')
ZIP = WORK / f'smoke_report_{stamp}.zip'
RUN2 = Path(R2['run_dir'])
names = ['manifest.json', 'metrics.json', 'translation_final.json', 'run_started.json', 'glossary.json',
         'course_memory_snapshot.json', 'candidates_scored.csv', 'refine_log.json']
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for run, tag in ((RUN1, 'run1'), (RUN2, 'run2_replay')):
        for n in names:
            if (run / n).exists():
                z.write(run / n, f'{tag}/{n}')
    for extra in (RUN1.parent / f'{RUN1.name}__smoke_check.json', RUN2.parent / f'{RUN2.name}__replay_check.json',
                  COURSE_MEMORY):
        if extra.exists():
            z.write(extra, extra.name)
    for p in sorted((WORK / 'smoke_logs').glob('*.log')):
        z.write(p, f'logs/{p.name}')
    for p in sorted((WORK / 'cache').rglob('*.failure.json')):
        z.write(p, f'failures/{p.relative_to(WORK / "cache")}')
    z.writestr('environment.json', json.dumps({'commit': COMMIT, 'preflight': PREFLIGHT, 'run1': {k: R1[k] for k in ('llm_requests', 'llm_request_cap', 'gpu_peak_gb')},
                                               'run2': {k: R2[k] for k in ('llm_requests', 'llm_request_cap', 'gpu_peak_gb')}},
                                              ensure_ascii=False, indent=1, default=str))
print('send this file:', ZIP)